In [75]:
# import warnings
# warnings.filterwarnings('ignore')

In [76]:
import os
import pprint
import tempfile

from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import tensorflow_recommenders as tfrs

In [77]:
# Load Dataset
ratings = tfds.load("movielens/100k-ratings", split="train")
movies = tfds.load("movielens/100k-movies", split="train")

In [78]:
for x in ratings.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'bucketized_user_age': 45.0,
 'movie_genres': array([7]),
 'movie_id': b'357',
 'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'timestamp': 879024327,
 'user_gender': True,
 'user_id': b'138',
 'user_occupation_label': 4,
 'user_occupation_text': b'doctor',
 'user_rating': 4.0,
 'user_zip_code': b'53211'}


2026-03-09 20:31:53.363854: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [79]:
for x in movies.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'movie_genres': array([4]),
 'movie_id': b'1681',
 'movie_title': b'You So Crazy (1994)'}


2026-03-09 20:31:53.402975: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [80]:
# user_id 및 movie_title 만
ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
})

movies = movies.map(lambda x: x["movie_title"])

In [81]:
print(len(ratings))
print(len(movies))

100000
1682


In [ ]:
# Train/ Test 분할
# ~~~~~~~~~~~~~SIX SEVEN~~~~~~~~~~~~~~~~~~~~~~~
tf.random.set_seed(67)
shuffled = ratings.shuffle(len(ratings), seed=67, reshuffle_each_iteration=False)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [83]:
print(len(train))
print(len(test))

80000
20000


---

데이터에 있는 고유한 사용자ID와 영화제목도 알아봅시다.

이는 범주형 기능(숫자 데이터가 아닌 글자 데이터)의 원시 값을 모델의 임베딩 벡터에 매핑할 수 있어야 하기 때문에 중요합니다.

그렇게 하려면 인접한 범위의 정수로 매핑하는 어휘가 필요합니다. 이를 통해 임베딩 테이블에서 임베딩을 조회할 수 있습니다.

In [84]:
# 중복제거
movie_titles = movies.batch(1_000)
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:10]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)',
       b'2 Days in the Valley (1996)',
       b'20,000 Leagues Under the Sea (1954)',
       b'2001: A Space Odyssey (1968)',
       b'3 Ninjas: High Noon At Mega Mountain (1998)',
       b'39 Steps, The (1935)'], dtype=object)

모델 구현

---

모델의 아키텍처를 설정하는 것은 모델링의 핵심 부분입니다.

2개의 타워 검색 모델을 구축하기 때문에 각 타워를 개별적으로 구축한 다음 최종 모델에서 결합할 수 있습니다.

1.쿼리 타워
2. 후보 타워

In [85]:
# 쿼리 타워
# --- 쿼리 및 후보 표현의 차원을 결정하기
embedding_dimension = 32 # 값이 높을수록 더 정확한 모델에 해당하지만 과적합된다.

# --- 모델 자체를 정의하기
user_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_user_ids, mask_token=None),
    tf.keras.layers.Embedding(len(unique_user_ids)+1, embedding_dimension)
])

In [87]:
# 후보 타워
movie_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_movie_titles, mask_token=None),
    tf.keras.layers.Embedding(len(unique_movie_titles)+1, embedding_dimension)
])

힘내! <img src="https://img1.daumcdn.net/thumb/R800x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FlcYNc%2Fbtq5L5jjbUT%2FAAAAAAAAAAAAAAAAAAAAAMqPmfOUGWqgvCH_iRsie_8of1JJWOMQJNgy1PqmSwzM%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1774969199%26allow_ip%3D%26allow_referer%3D%26signature%3DI6BAEoY9N93st8ouY50bu2CKFOM%253D" width="50%">

---

**tfrs.metrics.FactorizedTopK*** 메트릭사용.

- 상황: 유저1이 '토이스토리'를 봤음.

- 모델: 유저1은 '주토피아'가 1위고 '토이스토리'는 500위? (틀림)

- task의 반응: 벌점 100점! 유저1과 토이스토리 임베딩수를 서로 비슷하게 수정하삼!

- 결과: 모델이 숫자를 고쳐서 다음번에는 '토이스토리'를 10위 안으로 올리게 됨.ㅎㅎ

In [ ]:
# 측정 항목
metrics = tfrs.metrics.FactorizedTopK(
    candidates=movies.batch(128).map(movie_model) 
    # 유니크하게바뀐걸 넣어주는 게 정석인데 넘파이계열이라 ㄴㄴ 데이터셋을넣어야함
    # 영화 다 넣어서 32개숫자보따리(임베딩)로 바꿔놓으삼 --> 정답후보지
    # TopK의 역할: (훈련 중) 후보뽑아서 최소 K등 안에는 정답이 있나 체크 -> 너무못하면 loss크게줌.(아래)
)

# 손실(Loss)
task = tfrs.tasks.Retrieval(
    metrics=metrics
    # Retrieval의 역할: 정답 오답 구별/ 손실 계산-반환
    # 이를 사용해서 모델의 훈련 루프를 구현
)

| 데이터 소스 | 데이터 항목 | 사용되는 곳 |
| :--- | :--- | :--- |
| Ranting Dataset | user_id | user_model |
| Ranting Dataset | movie_title | movie_model |
| Movie Dataset | movie_title | metrics의 candidates |

- tfrs.Model 기본 클래스는 단순히 편리한 클래스임. (아우~ 추상적이야)
- tfrs.Model을 상속받음으로써 얻는 효과: 파편화된 쿼리/ 후보 타워/ Task를 하나로 합치기 위해~
- 표준 tf.keras.Model은 이런 모델 두 개 동시에 처리하는 복잡한 계산(유저랑 영화를 각각 임베딩해서 비교하기) 못해
- tfrs.Model 상속 받기만 하면 compute_loss함수를 모델이 '아이게 내 *공부방식*이군아'라고 알아먹음.

---

- 대조학습 (*공부방식*)

정답 데이터(P): 유저1이 '토이스토리'봤으면 두 임베딩 수가 서로 달라붙음 (내적점수를 높임)

오답 데이터(N): 유저1이 안 본 수천개의 다른 영화들은 -> 그 숫자들과는 서로 밀어냄. (내적 점수를 낮춤)

**평범하게쓸줄도알아야합니다.**

In [ ]:
# 전체 모델
class MovielensModel(tfrs.Model):
    def __init__(self, user_model, movie_model):
        super().__init__()
        self.movie_model: tf.keras.Model = movie_model
        self.user_model: tf.keras.Model = user_model
        self.task: tf.keras.layers.Layer = task
    
    
    # 모델이 한 번 공부할 때마다 이 함수를 실행해서 반성문 작성함.
    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor: 
        # -> 결과값으로 tf.Tensor형태의 숫자를 뱉어내겠다는 약속
        user_embeddings = self.user_model(features["user_id"])
        positive_movie_embeddings = self.movie_model(features["movie_title"])
        
        return self.task(user_embeddings, positive_movie_embeddings) # 두 변수 받아 점수내리기

In [ ]:
# 피팅 및 평가
